# Train **My Own AI Model** on Google Colab

Trains the from-scratch GPT (`llm-from-scratch/`) to completion on Colab's free compute — which runs a cell uninterrupted, unlike the ephemeral dev container. Produces a `web/model.json` you download and hand back to deploy to the live app at `/llm/`.

**How to use:** `Runtime → Run all` (or run each cell top to bottom). Training takes ~20–40 min. The last cell downloads `model.json`. **Keep the tab open** during training or Colab disconnects.

⚠️ This is a small model: it learns the *style* of the training text (Wikipedia → encyclopedic voice), not real facts. Names/dates it produces are invented.

The defaults are tuned so the result stays **readable and fast enough to run in a browser**: English-only corpus (so the vocabulary doesn't fill up with foreign scripts), a modest ~2.4M-param model, and enough steps to actually converge. A much bigger model produces a huge download and generates too slowly on a phone.

## 1. Get the code + dependencies

In [ ]:
!git clone --depth 1 https://github.com/Refayethossain28/BallrzAPP.git
%cd BallrzAPP/llm-from-scratch
!pip -q install numpy

## 2. Pick + clean a corpus

Default is **WikiText-2** (~10 MB of real Wikipedia article text). Swap `CORPUS_URL` for any plain-text UTF-8 file to change the voice. The cleanup strips WikiText markup (`<unk>`, `@,@`) **and non-English characters** — important, because Wikipedia is full of foreign scripts/symbols that otherwise fill the vocabulary and turn a small model's output into gibberish.

In [ ]:
CORPUS_URL = "https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/train.txt"
CLEAN_WIKITEXT = True   # set False for a corpus that is already plain English prose

import urllib.request, re, os
os.makedirs('data', exist_ok=True)
urllib.request.urlretrieve(CORPUS_URL, 'data/corpus.txt')
text = open('data/corpus.txt', encoding='utf-8', errors='ignore').read()
if CLEAN_WIKITEXT:
    text = text.replace('@,@', '').replace(' @.@ ', '.').replace(' @-@ ', '-').replace('<unk>', '')
    text = re.sub(r' ([,.;:!?)])', r'\1', text).replace('( ', '(')
# Keep printable ASCII only, so the tokenizer learns English — not 隊/ს/プ.
text = ''.join(ch for ch in text if ch == '\n' or 32 <= ord(ch) < 127)
text = re.sub(r'[ \t]{2,}', ' ', text)
open('data/corpus.txt', 'w', encoding='utf-8').write(text)
print(f'corpus: {len(text):,} chars')

## 3. Train

Defaults: a **6-layer / 160-dim (~2.4M param)** BPE model — bigger than the currently-deployed one but still fast enough in a browser, trained for 3000 steps. Resumable: if the Colab runtime drops, just run this cell again and it continues from the last checkpoint.

Want more quality and don't mind a slower/heavier model? Bump `--n_embd 192` or `--n_layer 8`. Want it faster/smaller? Lower `--steps` or `--n_embd`.

In [ ]:
!python train.py --data data/corpus.txt --tokenizer bpe --vocab_size 512 \
    --n_layer 6 --n_head 8 --n_embd 160 --block_size 96 --batch_size 16 \
    --steps 3000 --lr 3e-4 --min_lr_ratio 0.05 --eval_every 200 --out ckpt.npz

## 4. Export the browser weights and preview a sample

In [ ]:
!python export_web.py --ckpt ckpt.npz --out web/model.json
!python sample.py --ckpt ckpt.npz --prompt 'The history of ' --tokens 200 --temperature 0.7 --top_p 0.9 --repetition_penalty 1.3

## 5. Download `model.json`

Check the sample above reads like English first. Then send this file back to deploy it to the live app (or drop it into `llm-from-scratch/web/model.json` in the repo and push).

In [ ]:
from google.colab import files
print('model.json size:', os.path.getsize('web/model.json'), 'bytes')
files.download('web/model.json')